In [1]:
import gcsfs
import pandas as pd

from _utils import GCS_FILE_PATH
from process_ntd import *

## Loading Capital and Operating Funding and Expense Data 

In [2]:
df_funding = load_cap_op_funding()
df_capex = load_capex()
df_opex = load_service_and_opex()

In [3]:
df_funding.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'agency_status',
       'census_year', 'last_report_year', 'reporter_type', 'reporting_module',
       'uace_code', 'uza_area_sq_miles', 'primary_uza_name', 'uza_population',
       '_2024_status', 'source_agency', 'source_city', 'source_state',
       'operating_total', 'operating_federal', 'operating_state',
       'operating_local', 'operating_other', 'capital_total',
       'capital_federal', 'capital_state', 'capital_local', 'capital_other'],
      dtype='object')

In [4]:
df_capex.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'mode', 'mode_full_name',
       'agency_status', 'census_year', 'last_report_year', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles', 'uza_name',
       'uza_population', 'total_capital_expenditures',
       'rolling_stock_expenditures', 'facilities_expenditures',
       'other_expenditures', '_2024_mode_status', 'source_agency',
       'source_city', 'source_state'],
      dtype='object')

In [5]:
df_opex.columns

Index(['key', 'ntd_id', 'mode', 'year', 'type_of_service',
       'unlinked_passenger_trips', 'vehicle_revenue_hours',
       'vehicle_revenue_miles', 'vehicles_operated_in_maxiumum_service',
       'passenger_miles_traveled', 'directional_route_miles',
       'operating_expenses_vehicle_operations',
       'operating_expenses_vehicle_maintenance',
       'operating_expenses_nonvehicle_maintenance',
       'operating_expenses_general_administration', 'operating_expenses_total',
       'fare_revenue', 'opex_per_vrh', 'opex_per_vrm', 'opex_per_upt',
       'upt_per_vrh', 'upt_per_vrm', 'farebox_recovery_ratio', 'agency_status',
       'census_year', 'last_report_year', 'mode_status', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles',
       'primary_uza_name', 'uza_population', 'source_agency', 'source_city',
       'source_state', 'upt_prior_year', 'upt_change_1yr',
       'upt_pct_change_1yr', 'mode_full_name', 'type_of_service_full_name',
       'service_typ

In [6]:
opex_cols = [
    "key", "ntd_id", "year", "mode", "mode_full_name", "agency_status", "census_year", "last_report_year",
    "reporter_type", "reporting_module", "uace_code", "uza_area_sq_miles", "primary_uza_name", "uza_population",
    "operating_expenses_total", "operating_expenses_vehicle_operations", "operating_expenses_vehicle_maintenance",
    "operating_expenses_nonvehicle_maintenance", "operating_expenses_general_administration",
    "source_agency", "source_city", "source_state", "type_of_service", "type_of_service_full_name", "service_type"]

In [7]:
df_opex = df_opex[opex_cols]

## Loading BLS Consumer Price Index Data

In [8]:
cpi_annual = get_annual_average_cpi(2015, 2024)

cpi_annual.head(5)

,year,cpi
0,2015,237.017000
1,2016,240.007167
2,2017,245.119583
3,2018,251.106833
4,2019,255.657417


In [9]:
base_cpi = cpi_annual.loc[cpi_annual['year'] == 2024, 'cpi'].iloc[0]

## New Funding and Expenses Related Columns

In [10]:
df_funding = calculate_shares(
    df_funding,
    total_col='operating_total',
    source_cols={
        'federal': 'operating_federal',
        'state': 'operating_state',
        'local': 'operating_local',
        'other': 'operating_other'
    },
    suffix='op'
)

df_funding = calculate_shares(
    df_funding,
    total_col='capital_total',
    source_cols={
        'federal': 'capital_federal',
        'state': 'capital_state',
        'local': 'capital_local',
        'other': 'capital_other'
    },
    suffix='cap'
)

In [11]:
df_capex = calculate_shares(
    df_capex,
    total_col="total_capital_expenditures",
    source_cols={
        "rolling_stock": "rolling_stock_expenditures",
        "facilities": "facilities_expenditures",
        "other": "other_expenditures",
    },
    suffix="cap",
)

df_opex = calculate_shares(
    df_opex,
    total_col="operating_expenses_total",
    source_cols={
        "vehicle_operations": "operating_expenses_vehicle_operations",
        "vehicle_maintenance": "operating_expenses_vehicle_maintenance",
        "nonvehicle_maintenance": "operating_expenses_nonvehicle_maintenance",
        "general_administration": "operating_expenses_general_administration",
    },
    suffix="opex",
)

## 1. Funding Characteristics & Analysis

#### 1A. Real Capital and Operating Funding Values

In [12]:
df_funding_wd_real = add_real_values(
    df_funding,
    value_cols={
        "capital_total": "capital_total_real",
        "operating_total": "operating_total_real"
    },
    cpi_annual=cpi_annual,
    base_cpi=base_cpi
)


#### 1B. Herfindahl-style concentration index (HHI)

The Herfindahl-style concentration index (HHI) measures how concentrated an agency’s  funding is across the four funding sources: federal, state, local, and other. It is calculated by squaring each source’s share of total operating and capital funding funding and summing the squared shares. The index ranges from 0.25 to 1.00 when there are four funding sources: a value of 0.25 indicates that funding is evenly distributed across all four sources (25% each), while a value of 1.00 indicates that the agency receives all of its operating funding from a single source. Thus, higher HHI values indicate greater dependence on a small number of funding sources and potentially greater fiscal concentration risk, while lower values indicate a more diversified funding structure.

In [14]:
cap_share_cols = [
    'federal_cap_share',
    'state_cap_share',
    'local_cap_share',
    'other_cap_share'
]

df_funding_wd_real['capital_concentration'] = (
    df_funding_wd_real[cap_share_cols].pow(2).sum(axis=1, min_count=1)
)

op_share_cols = [
    'federal_op_share',
    'state_op_share',
    'local_op_share',
    'other_op_share'
]

df_funding_wd_real['operating_concentration'] = (
    df_funding_wd_real[op_share_cols].pow(2).sum(axis=1, min_count=1)
)

#### 1C. Federal Capital Tilt

Federal Capital Tilt measures whether an agency relies more heavily on federal funding for capital investments than for day-to-day operating expenses. A positive value means federal funding plays a larger role in capital funding than operating funding, while a negative value means the agency is more federally dependent for operations.

In [15]:
df_funding_wd_real['federal_capital_tilt'] = df_funding_wd_real['federal_cap_share'] - df_funding_wd_real['federal_op_share']

#### 1D. Federal Operating Leverage

Federal Operating Leverage measures how much an agency’s total operating funding is supported by federal funding. A higher value indicates greater reliance on federal resources for ongoing operations, while a lower value indicates that the agency’s operations are funded primarily through state, local, or other sources.

In [16]:
df_funding_wd_real['federal_operating_leverage'] = (
    df_funding_wd_real['operating_federal'] / df_funding_wd_real['operating_total']
)

#### 1E. Local Operating Funding Intensity

How many local dollars each resident of the service area contributes to operating funding. Measures local fiscal effort/burden per capita?

In [17]:
df_funding_wd_real['local_operating_funding_per_uza_resident'] = df_funding_wd_real['operating_local'] / df_funding_wd_real['uza_population']

#### 1F. Operating Funding Burden per Capita

Total operating funding (all sources) divided by service-area population. Measures overall funding intensity per resident.

In [18]:
df_funding_wd_real['operating_total_per_capita'] = df_funding_wd_real['operating_total']/ df_funding_wd_real['uza_population']

## 2. Expenditure Characteristics & Analysis

Because Funding is already at the agency-year (ntd_id × year) level, CapEx and OpEx need to be brought to that same level before they are merged. Since OpEx contains multiple modes and, as we discovered, multiple type_of_service records within a mode, we should aggregate the raw expenditure amounts across all modes/service types first. We will then calculate the indices later, after CPI and other required variables are merged in. For now, the goal is simply to create clean one-row-per-agency-year CapEx and OpEx datasets.

In [19]:
df_opex_agency = aggregate_opex_to_agency_year(df_opex)

In [20]:
df_capex_agency = aggregate_capex_to_agency_year(df_capex)

#### 2A. Real Capital and Operating Expenditure Values

In [ ]:
df_capex_wd_real = add_real_values(
    df_capex_agency,
    value_cols={
        "total_capital_expenditures": "capex_total_real"
    },
    cpi_annual=cpi_annual,
    base_cpi=base_cpi
)

df_opex_wd_real = add_real_values(
    df_opex_agency,
    value_cols={
        "operating_expenses_total": "opex_total_real"
    },
    cpi_annual=cpi_annual,
    base_cpi=base_cpi
)

#### 2B. Herfindahl-style concentration index (HHI)

In [20]:
# Capex concentration — 3 categories, range 0.333–1.00
cap_exp_share_cols = [
    'rolling_stock_cap_share',
    'facilities_cap_share',
    'other_cap_share'
]

df_capex_wd_real['capital_expense_concentration'] = (
    df_capex_wd_real[cap_exp_share_cols].pow(2).sum(axis=1, min_count=1)
)

# Opex concentration — 4 categories, range 0.25–1.00
op_exp_share_cols = [
    'vehicle_operations_opex_share',
    'vehicle_maintenance_opex_share',
    'nonvehicle_maintenance_opex_share',
    'general_administration_opex_share'
]

df_opex_wd_real['operating_expense_concentration'] = (
    df_opex_wd_real[op_exp_share_cols].pow(2).sum(axis=1, min_count=1)
)

#### 2C. Capital and Operating Spending Per Capita

How much capital investment is happening relative to the size of the population served?

In [21]:
df_capex_wd_real['capital_expenditure_per_capita'] = (
    df_capex_wd_real['total_capital_expenditures'] / df_capex_wd_real['uza_population']
)

df_opex_wd_real['operating_expense_per_capita'] = (
    df_opex_wd_real['operating_expenses_total'] / df_opex_wd_real['uza_population']
)

#### 2D. Capital and Operating Spending Volatility

Rolling coefficient of variation (std/mean) of capital spending over a multi-year window per agency. Measures how erratic vs. steady an agency's capital investment pattern is over time.

In [22]:
df_capex_wd_real['capex_volatility'] = (
    df_capex_wd_real.groupby('ntd_id')['total_capital_expenditures']
    .transform(lambda x: x.rolling(4, min_periods=3).std() / x.rolling(4, min_periods=3).mean())
)

df_opex_wd_real['opex_volatility'] = (
    df_opex_wd_real.groupby('ntd_id')['operating_expenses_total']
    .transform(lambda x: x.rolling(4, min_periods=3).std() / x.rolling(4, min_periods=3).mean())
)

#### 2E. Real Capex and Opex Growth Rate (YoY, CPI-adjusted)

Year-over-year percent change in capital spending, using inflation-adjusted (real) dollars.

In [23]:
df_capex_wd_real['real_capex_growth'] = (
    df_capex_wd_real.groupby('ntd_id')['capex_total_real']
    .pct_change(fill_method=None)
)

df_opex_wd_real['real_opex_growth'] = (
    df_opex_wd_real.groupby('ntd_id')['opex_total_real']
    .pct_change(fill_method=None)
)

In [24]:
df_funding_merged.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'agency_status',
       'census_year', 'last_report_year', 'reporter_type', 'reporting_module',
       'uace_code', 'uza_area_sq_miles', 'primary_uza_name', 'uza_population',
       '_2024_status', 'source_agency', 'source_city', 'source_state',
       'operating_total', 'operating_federal', 'operating_state',
       'operating_local', 'operating_other', 'capital_total',
       'capital_federal', 'capital_state', 'capital_local', 'capital_other',
       'federal_op_share', 'state_op_share', 'local_op_share',
       'other_op_share', 'federal_cap_share', 'state_cap_share',
       'local_cap_share', 'other_cap_share', 'cpi', 'operating_total_real',
       'capital_total_real', 'capital_concentration',
       'operating_concentration', 'federal_capital_tilt',
       'federal_operating_leverage',
       'local_operating_funding_per_uza_resident',
       'operating_total_per_capita'],
      dtype='object')

In [25]:
df_capex_wd_real.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'mode', 'mode_full_name',
       'agency_status', 'census_year', 'last_report_year', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles', 'uza_name',
       'uza_population', 'total_capital_expenditures',
       'rolling_stock_expenditures', 'facilities_expenditures',
       'other_expenditures', '_2024_mode_status', 'source_agency',
       'source_city', 'source_state', 'rolling_stock_cap_share',
       'facilities_cap_share', 'other_cap_share', 'cpi', 'capex_total_real',
       'capital_expense_concentration', 'capital_expenditure_per_capita',
       'capex_volatility', 'real_capex_growth'],
      dtype='object')

In [26]:
df_opex_wd_real.columns

Index(['key', 'ntd_id', 'year', 'mode', 'mode_full_name', 'agency_status',
       'census_year', 'last_report_year', 'reporter_type', 'reporting_module',
       'uace_code', 'uza_area_sq_miles', 'primary_uza_name', 'uza_population',
       'operating_expenses_total', 'operating_expenses_vehicle_operations',
       'operating_expenses_vehicle_maintenance',
       'operating_expenses_nonvehicle_maintenance',
       'operating_expenses_general_administration', 'source_agency',
       'source_city', 'source_state', 'vehicle_operations_opex_share',
       'vehicle_maintenance_opex_share', 'nonvehicle_maintenance_opex_share',
       'general_administration_opex_share', 'cpi', 'opex_total_real',
       'operating_expense_concentration', 'operating_expense_per_capita',
       'opex_volatility', 'real_opex_growth'],
      dtype='object')

## Combining Funding and Expenditure Data to Create New Indices

The indices above are calculated separately for capital funding, operating funding, capital expenditures, and operating expenditures.

In [24]:
df_combined = (
    df_funding
    .merge(
        df_capex_agency,
        on=["ntd_id", "year"],
        how="inner",
        suffixes=("_funding", "_capex")
    )
    .merge(
        df_opex_agency,
        on=["ntd_id", "year"],
        how="inner",
        suffixes=("", "_opex")
    )
)

In [25]:
print("Rows:", len(df_combined))
print(
    "Unique ntd_id × year:",
    df_combined[["ntd_id", "year"]].drop_duplicates().shape[0]
)
print(
    "Duplicate ntd_id × year:",
    df_combined.duplicated(["ntd_id", "year"]).sum()
)


Rows: 3520
Unique ntd_id × year: 3520
Duplicate ntd_id × year: 0


In [ ]:
df_capex_merged = df_capex_agency.merge(
    cpi_annual,
    on='year',
    how='left'
)